# Trace Count v24.5: 20-set maximum-entropy control

This is a strict supervision-density rerun of v24.4. The paired RoPE models,
component-normalized loss, 256-character Shakespeare context, separator/no-index
trace, atomic answers, count support 1–10, seed, optimizer, maximum-entropy
set/count sampler, and 10,000-step schedule are unchanged.

The only substantive change is reducing the needle pool from 100 sets to 20.
Each semantic marker therefore receives roughly five times as many training
examples, while the sampler still enforces a uniform 5% set marginal and 10%
count marginal. This tests whether v24.4 failed because marker-specific
supervision was too sparse rather than because the no-index counter is absent.

Both modes are retrained and evaluated with the same held-out TF, exact-trace
conditional readout, count-confusion, phase, causal, and NCC diagnostics.


## 1. Mount Google Drive

In [ ]:
from pathlib import Path

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
DRIVE_READY = False
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_READY = True
    print("Drive ready:", DRIVE_RESULTS_ROOT)
else:
    print("Local runtime: Drive mount skipped")

## 2. Clone/update the repo on local Colab storage

In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

assert DRIVE_READY, "Run the Drive cell first"
REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
REPO_REF = "agent/remove-misplaced-realistic-artifacts"
preferred = Path("/content/Synthetic_CoT_NiaH_Count")
candidates = [Path.cwd(), *Path.cwd().parents, preferred]
repo = next((path.resolve() for path in candidates if (path / "pyproject.toml").exists()), None)
if repo is None:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(preferred)],
        check=True,
    )
    repo = preferred
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", REPO_REF], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "pull", "--ff-only", "origin", REPO_REF],
        check=True,
    )
assert (repo / "src" / "synthetic_counting_v24_5").is_dir(), f"synthetic_counting_v24_5 is absent from {repo}"
os.chdir(repo)

scientific_probe = subprocess.run(
    [sys.executable, "-c", "import numpy,pandas,scipy,matplotlib,seaborn"],
    capture_output=True,
    text=True,
)
if scientific_probe.returncode:
    print(scientific_probe.stderr[-2000:])
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "--force-reinstall", "numpy==1.26.4", "pandas==2.2.3",
            "scipy==1.13.1", "matplotlib==3.8.4", "seaborn==0.13.2",
        ],
        check=True,
    )
    if Path("/content").exists():
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Scientific ABI repaired. Reconnect and rerun all cells.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
src_root = str(repo / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
os.environ["PYTHONPATH"] = src_root + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np
import pandas as pd
import torch
import synthetic_counting_v24_5
from IPython.display import Image, display

def run_streaming(command):
    import codecs
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        print(decoder.decode(chunk), end="", flush=True)
    print(decoder.decode(b"", final=True), end="", flush=True)
    returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

repo_commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
print({
    "repo": str(repo),
    "repo_ref": REPO_REF,
    "repo_commit": repo_commit,
    "package": str(Path(synthetic_counting_v24_5.__file__).resolve()),
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

## 3. Auditable pool-size-only contrast

V24.5 differs from v24.4 only in `needle_pool_size: 100 -> 20` (plus the
version label). Counts 1–10, maximum-entropy sampling, loss coefficients, model,
seed, and schedule remain fixed.

For example, with 100 sets and 10 counts, 320,000 task examples provide about
320 examples per set/count cell before feasibility corrections. With 20 sets,
the same budget provides about 1,600 examples per cell. The counter must still
retrieve the queried marker, but it sees enough repeats to learn a stable
marker-invariant counting and answer readout rule.


In [ ]:
VERSION = "v24.5"
TRACE_FORMAT = "separator"
PRESET = "main"                 # fixed: count 1-10 requires the matched 256-char setting
SEED = 1234
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COUNT_MAX_THRESHOLD = 10
NEEDLE_POOL_SIZE = 20
NEEDLE_POOL_FREQUENCY_THRESHOLD = 10.0 / 256.0
TASK_OCCURRENCE_RATIO = 1.0
TRAINING_COUNT_DISTRIBUTION = "maxent_set_count"
TASK_OUTPUT_LOSS_REDUCTION = "component_normalized"
MAX_TRAIN_STEPS = 10_000
MAX_STEPS_FOR_LANGUAGE_PRED = 1_500
CHECKPOINT_EVERY_STEPS = 100     # model-only scientific snapshot
RECOVERY_EVERY_STEPS = 500       # full optimizer/RNG recovery state
SNAPSHOT_SHARD_EVERY_STEPS = 500 # five 100-step snapshots per file
EVAL_EVERY_STEPS = 500
AR_EVAL_EVERY_STEPS = 1_000
AR_EXAMPLES_PER_COUNT = 2
PERMUTATION_EXAMPLES_PER_COUNT = 1  # each expands to all six query orders
EVAL_EXAMPLES_PER_COUNT = 10
FINAL_EXAMPLES_PER_COUNT = 50
PHASE_SELECTION_EXAMPLES_PER_COUNT = 2
PHASE_REPORT_EXAMPLES_PER_COUNT = 1
if PRESET == "debug":
    MAX_TRAIN_STEPS = 6
    MAX_STEPS_FOR_LANGUAGE_PRED = 6
    CHECKPOINT_EVERY_STEPS = 3
    RECOVERY_EVERY_STEPS = 3
    SNAPSHOT_SHARD_EVERY_STEPS = 3
    EVAL_EVERY_STEPS = 3
    AR_EVAL_EVERY_STEPS = 3
    AR_EXAMPLES_PER_COUNT = 1
    EVAL_EXAMPLES_PER_COUNT = 2
    FINAL_EXAMPLES_PER_COUNT = 2
    PHASE_SELECTION_EXAMPLES_PER_COUNT = 1
OUT_ROOT = "runs/synthetic_counting_v24_5"
RUN_NAME = "v24.5_pool20_maxent_count1-10_seed1234"
SKIP_COMPLETED = True
AUTO_DISCONNECT = True
DISCONNECT_DELAY_SECONDS = 10
# The run folder itself is the Drive result folder; no second checkpoint copy.
CHECKPOINT_SYNC_ROOT = DRIVE_RESULTS_ROOT

from synthetic_counting_v24_5.config import preset_config
PLANNED_CONFIG = preset_config(
    PRESET,
    seed=SEED,
    device=DEVICE,
    count_max_threshold=COUNT_MAX_THRESHOLD,
    needle_pool_size=NEEDLE_POOL_SIZE,
    needle_pool_frequency_threshold=NEEDLE_POOL_FREQUENCY_THRESHOLD,
    task_occurrence_ratio=TASK_OCCURRENCE_RATIO,
    training_count_distribution=TRAINING_COUNT_DISTRIBUTION,
    task_output_loss_reduction=TASK_OUTPUT_LOSS_REDUCTION,
    train_steps=MAX_TRAIN_STEPS,
    max_steps_for_language_pred=MAX_STEPS_FOR_LANGUAGE_PRED,
    checkpoint_every=CHECKPOINT_EVERY_STEPS,
    recovery_every=RECOVERY_EVERY_STEPS,
    snapshot_shard_every=SNAPSHOT_SHARD_EVERY_STEPS,
    eval_every=EVAL_EVERY_STEPS,
    ar_eval_every=AR_EVAL_EVERY_STEPS,
    ar_examples_per_count=AR_EXAMPLES_PER_COUNT,
    permutation_examples_per_count=PERMUTATION_EXAMPLES_PER_COUNT,
    eval_examples_per_count=EVAL_EXAMPLES_PER_COUNT,
    final_examples_per_count=FINAL_EXAMPLES_PER_COUNT,
    phase_head_selection_examples_per_count=PHASE_SELECTION_EXAMPLES_PER_COUNT,
    phase_examples_per_count=PHASE_REPORT_EXAMPLES_PER_COUNT,
)
print(PLANNED_CONFIG.to_dict())
from dataclasses import asdict
from synthetic_counting_v24_4.config import preset_config as v24_4_preset_config
V24_4_BASELINE_CONFIG = v24_4_preset_config(PRESET, seed=SEED, device=DEVICE)
changed_fields = {
    key for key, value in asdict(PLANNED_CONFIG).items()
    if asdict(V24_4_BASELINE_CONFIG).get(key) != value
}
assert changed_fields == {"version", "needle_pool_size"}, changed_fields
print("Controlled difference from v24.4:", sorted(changed_fields))
assert PLANNED_CONFIG.trace_format == TRACE_FORMAT
assert PLANNED_CONFIG.version == VERSION
assert PLANNED_CONFIG.training_count_distribution == TRAINING_COUNT_DISTRIBUTION
assert PLANNED_CONFIG.task_output_loss_reduction == TASK_OUTPUT_LOSS_REDUCTION
assert PLANNED_CONFIG.enabled_model_variants == (
    "rope/nonthinking", "rope/thinking"
)
assert PLANNED_CONFIG.count_max_threshold == 10
assert PLANNED_CONFIG.needle_pool_frequency_threshold == 10.0 / 256.0
assert PLANNED_CONFIG.needle_pool_size == NEEDLE_POOL_SIZE
assert PLANNED_CONFIG.final_count_loss_weight == 1.0
assert PLANNED_CONFIG.cot_trace_loss_weight == 1.0
assert PLANNED_CONFIG.task_output_count_weight == 1.0
assert PLANNED_CONFIG.task_output_trace_weight == 1.0
assert PLANNED_CONFIG.task_output_structure_weight == 0.1
print({
    "sequence_layout": "<BOS> query[5] data[256] output",
    "number_representation": PLANNED_CONFIG.count_tokenization,
    "trace_format": PLANNED_CONFIG.trace_format,
    "thinking_trace": "(<Sep> marker) repeated n times",
    "max_render_len": PLANNED_CONFIG.max_render_len,
    "dense_snapshots_per_model": len(range(0, MAX_TRAIN_STEPS + 1, CHECKPOINT_EVERY_STEPS)),
    "snapshot_files_per_model": MAX_TRAIN_STEPS // SNAPSHOT_SHARD_EVERY_STEPS + 1,
    "recovery_policy": "rolling latest plus pinned objective boundary and final",
})

## 4. Prepare fixed data, pool, and evaluation manifests

In [ ]:
base_cmd = [
    sys.executable, "-u", "-m", "synthetic_counting_v24_5.run_v24_5",
    "--preset", PRESET,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--count-max-threshold", str(COUNT_MAX_THRESHOLD),
    "--needle-pool-size", str(NEEDLE_POOL_SIZE),
    "--needle-pool-frequency-threshold", str(NEEDLE_POOL_FREQUENCY_THRESHOLD),
    "--task-occurrence-ratio", str(TASK_OCCURRENCE_RATIO),
    "--training-count-distribution", TRAINING_COUNT_DISTRIBUTION,
    "--task-output-loss-reduction", TASK_OUTPUT_LOSS_REDUCTION,
    "--train-steps", str(MAX_TRAIN_STEPS),
    "--max-steps-for-language-pred", str(MAX_STEPS_FOR_LANGUAGE_PRED),
    "--checkpoint-every", str(CHECKPOINT_EVERY_STEPS),
    "--recovery-every", str(RECOVERY_EVERY_STEPS),
    "--snapshot-shard-every", str(SNAPSHOT_SHARD_EVERY_STEPS),
    "--eval-every", str(EVAL_EVERY_STEPS),
    "--ar-eval-every", str(AR_EVAL_EVERY_STEPS),
    "--ar-examples-per-count", str(AR_EXAMPLES_PER_COUNT),
    "--permutation-examples-per-count", str(PERMUTATION_EXAMPLES_PER_COUNT),
    "--eval-examples-per-count", str(EVAL_EXAMPLES_PER_COUNT),
    "--final-examples-per-count", str(FINAL_EXAMPLES_PER_COUNT),
    "--phase-head-selection-examples-per-count", str(PHASE_SELECTION_EXAMPLES_PER_COUNT),
    "--phase-examples-per-count", str(PHASE_REPORT_EXAMPLES_PER_COUNT),
    "--out-root", OUT_ROOT,
    "--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT),
]
if RUN_NAME is not None:
    base_cmd += ["--run-name", RUN_NAME]
if SKIP_COMPLETED:
    base_cmd.append("--skip-completed")
run_streaming([*base_cmd, "--stage", "prepare"])

from synthetic_counting_v20.config import default_run_name
RUN_DIR = Path(OUT_ROOT) / (RUN_NAME or default_run_name(PLANNED_CONFIG))
DRIVE_RUN_DIR = CHECKPOINT_SYNC_ROOT / RUN_DIR.name
print("RUN_DIR:", RUN_DIR.resolve())
print("DRIVE_RUN_DIR:", DRIVE_RUN_DIR)

## 5. Train the paired 20-set control

Non-thinking and Thinking are trained sequentially from the same seed. They
receive the same 20-set/count cell draws, corpus windows, set orders, model
initialization, optimizer, and schedule. Only their output grammar differs.


In [ ]:
training_started = time.perf_counter()
run_streaming([*base_cmd, "--stage", "train"])
print(f"Training block: {time.perf_counter() - training_started:.1f} seconds")

## 6. Dense phase, geometry, and local causal analyses

In [ ]:
analysis_started = time.perf_counter()
analysis_stages = "phase,plots"
run_streaming([*base_cmd, "--stage", analysis_stages])

NCC_OUTPUT = RUN_DIR / "analysis" / "aligned_ncc"
run_streaming([
    sys.executable, "-u", "scripts/compare_v24_modes_ncc.py",
    "--results-root", str(RUN_DIR.parent),
    "--output", str(NCC_OUTPUT),
    "--run-prefix", RUN_DIR.name,
    "--expected-version", VERSION,
    "--device", DEVICE,
    "--discovery-per-label", "10",
    "--confirmation-per-label", "8",
    "--batch-size", "32",
])
print(f"Analysis block: {time.perf_counter() - analysis_started:.1f} seconds")

expected = [
    RUN_DIR / "analysis" / "phase_transition" / "manifest.json",
    RUN_DIR / "analysis" / "phase_transition" / "interactive_manifold_3d.html",
    RUN_DIR / "analysis" / "phase_transition" / "tables" / "phase_transition_candidates.csv",
    RUN_DIR / "tables" / "training_token_exposure_by_k.csv",
    RUN_DIR / "tables" / "training_sampling_distribution.csv",
    RUN_DIR / "tables" / "training_set_count_sampler_plan.csv",
    RUN_DIR / "figures" / "dense_fixed_head_emergence.png",
    RUN_DIR / "figures" / "dense_marker_manifold_emergence.png",
    RUN_DIR / "figures" / "milestone_local_head_causality.png",
    RUN_DIR / "analysis" / "aligned_ncc" / "selected_confirmation_summary.csv",
]
missing = [str(path) for path in expected if not path.exists()]
assert not missing, "Missing required outputs: " + str(missing)
for path in expected:
    print(path.relative_to(RUN_DIR), path.stat().st_size)

## 7. Inspect the main diagnostics

In [ ]:
for filename in (
    "training_token_exposure_by_k.png",
    "dense_phase_behavior_by_count.png",
    "dense_fixed_head_emergence.png",
    "dense_marker_manifold_emergence.png",
    "milestone_local_head_causality.png",
):
    path = RUN_DIR / "figures" / filename
    if path.exists():
        display(Image(filename=str(path)))
display(pd.read_csv(RUN_DIR / "analysis" / "phase_transition" / "tables" / "fixed_head_rankings.csv").head(12))
display(pd.read_csv(RUN_DIR / "analysis" / "phase_transition" / "tables" / "milestone_local_head_causality.csv"))
display(pd.read_csv(NCC_OUTPUT / "selected_confirmation_summary.csv")[[
    "comparison_mode", "endpoint", "layer",
    "chance_balanced_accuracy",
    "confirmation_logistic_balanced_accuracy",
    "confirmation_ncc_balanced_accuracy",
    "confirmation_ncc_above_chance",
]])

sampling = pd.read_csv(RUN_DIR / "tables" / "training_sampling_distribution.csv")
accepted = sampling[sampling["dimension"].eq("accepted_counts")].copy()
accepted["count"] = accepted["value"].astype(int)
accepted["training_share"] = accepted["examples"] / accepted["task_examples"]
accepted = accepted.sort_values(["mode", "count"])
assert accepted.groupby("mode")["count"].nunique().eq(10).all()
assert (accepted["training_share"] - 0.1).abs().max() < 0.005
display(accepted[["mode", "count", "examples", "training_share"]])
display(pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_summary.csv"))
display(pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_by_count.csv"))

metrics = pd.read_csv(RUN_DIR / "tables" / "train_metrics.csv")
metrics["component_reduction_active"] = metrics["component_reduction_active"].astype(str).str.lower().eq("true")
pre = metrics[metrics["step"].le(MAX_STEPS_FOR_LANGUAGE_PRED)]
post = metrics[metrics["step"].gt(MAX_STEPS_FOR_LANGUAGE_PRED)]
assert not pre["component_reduction_active"].any()
assert post["component_reduction_active"].all()
last_post = post.sort_values("step").groupby("mode", as_index=False).tail(1)
expected_count_share = {"nonthinking": 1.0 / 1.1, "thinking": 1.0 / 2.1}
expected_trace_share = {"nonthinking": 0.0, "thinking": 1.0 / 2.1}
for row in last_post.itertuples(index=False):
    assert abs(row.batch_final_count_region_coefficient_share - expected_count_share[row.mode]) < 1e-9
    assert abs(row.batch_trace_region_coefficient_share - expected_trace_share[row.mode]) < 1e-9
display(last_post[[
    "mode", "step", "train_total_loss", "gradient_norm",
    "batch_final_count_token_weight_share",
    "batch_final_count_region_coefficient_share",
    "batch_trace_region_coefficient_share",
    "batch_structure_region_coefficient_share",
    "train_objective_final_count_region_mean_loss",
    "train_objective_trace_region_mean_loss",
    "train_objective_structure_region_mean_loss",
]])


sampler_plan = pd.read_csv(RUN_DIR / "tables" / "training_set_count_sampler_plan.csv")
target_set = sampler_plan.groupby("set_id", as_index=False)["target_probability"].sum()
target_count = sampler_plan.groupby("count", as_index=False)["target_probability"].sum()
assert (target_set["target_probability"] - 0.05).abs().max() < 1e-9
assert (target_count["target_probability"] - 0.10).abs().max() < 1e-9

set_exposure = sampling[sampling["dimension"].eq("set_ids")].copy()
set_exposure["training_share"] = set_exposure["examples"] / set_exposure["task_examples"]
assert set_exposure.groupby("mode")["value"].nunique().eq(20).all()
assert (set_exposure["training_share"] - 0.05).abs().max() < 0.001

final_summary = pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_summary.csv")
final_by_count = pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_by_count.csv")
final_detail = pd.read_csv(RUN_DIR / "tables" / "final_autoregressive_detail.csv")
eval_detail = pd.read_csv(RUN_DIR / "tables" / "eval_detail.csv")

thinking_summary = final_summary[final_summary["mode"].eq("thinking")].iloc[-1]
thinking_by_count = final_by_count[final_by_count["mode"].eq("thinking")].sort_values("count")
exact_trace = final_detail[
    final_detail["mode"].eq("thinking") & final_detail["trace_exact"].eq(1.0)
]
conditional_answer_accuracy_given_exact_trace = (
    exact_trace.groupby("count", as_index=False)["ar_accuracy"].mean()
)
readout_confusion = pd.crosstab(
    final_detail.loc[final_detail["mode"].eq("thinking"), "count"],
    final_detail.loc[final_detail["mode"].eq("thinking"), "ar_pred_count"],
    normalize="index",
)
heldout_tf = (
    eval_detail[
        eval_detail["mode"].eq("thinking")
        & eval_detail["step"].eq(MAX_TRAIN_STEPS)
    ]
    .groupby("count", as_index=False)["tf_final_accuracy"]
    .mean()
)

overall_accuracy = float(thinking_summary["ar_final_accuracy"])
minimum_count_accuracy = float(thinking_by_count["ar_final_accuracy"].min())
count_accuracy_spread = float(
    thinking_by_count["ar_final_accuracy"].max()
    - thinking_by_count["ar_final_accuracy"].min()
)
trace_exact_accuracy = float(thinking_summary["trace_exact"])
success_criteria_met = bool(
    overall_accuracy >= 0.90
    and minimum_count_accuracy >= 0.85
    and count_accuracy_spread <= 0.10
    and trace_exact_accuracy >= 0.90
)
print({
    "success_criteria_met": success_criteria_met,
    "overall_accuracy": overall_accuracy,
    "minimum_count_accuracy": minimum_count_accuracy,
    "count_accuracy_spread": count_accuracy_spread,
    "trace_exact_accuracy": trace_exact_accuracy,
})
display(target_set.describe())
display(target_count)
display(set_exposure.groupby("mode")["training_share"].describe())
display(thinking_by_count[["count", "ar_final_accuracy", "trace_exact"]])
display(heldout_tf)
display(conditional_answer_accuracy_given_exact_trace)
display(readout_confusion)


trace_readout_summary = pd.read_csv(RUN_DIR / "tables" / "trace_readout_summary.csv")
trace_readout_by_count = pd.read_csv(RUN_DIR / "tables" / "trace_readout_by_count.csv")
thinking_trace_readout = trace_readout_summary[
    trace_readout_summary["mode"].eq("thinking")
].iloc[-1]
trace_readout_success_criteria_met = str(
    thinking_trace_readout["success_criteria_met"]
).strip().lower() in {"true", "1", "1.0"}
print({
    "trace_readout_success_criteria_met": trace_readout_success_criteria_met,
    "trace_readout_accuracy": float(thinking_trace_readout["trace_readout_accuracy"]),
    "minimum_count_accuracy": float(thinking_trace_readout["minimum_count_accuracy"]),
    "count_accuracy_spread": float(thinking_trace_readout["count_accuracy_spread"]),
    "raw_ar_accuracy": float(thinking_trace_readout["raw_ar_accuracy"]),
    "trace_exact": float(thinking_trace_readout["trace_exact"]),
})
display(trace_readout_by_count[[
    "count", "raw_ar_accuracy", "trace_readout_accuracy",
    "trace_readout_answer_rate", "trace_exact",
]])


## 8. Verify Drive persistence and optionally disconnect

In [ ]:
# Re-running plots also triggers the pipeline's incremental final Drive sync.
run_streaming([*base_cmd, "--stage", "plots"])
required_drive_files = [
    DRIVE_RUN_DIR / "config.json",
    DRIVE_RUN_DIR / "manifest.json",
    DRIVE_RUN_DIR / "analysis" / "phase_transition" / "manifest.json",
    DRIVE_RUN_DIR / "checkpoints" / "rope" / "nonthinking" / "snapshot_index.csv",
    DRIVE_RUN_DIR / "checkpoints" / "rope" / "thinking" / "snapshot_index.csv",
    DRIVE_RUN_DIR / "checkpoints" / "rope" / "nonthinking" / "final" / "checkpoint.pt",
    DRIVE_RUN_DIR / "checkpoints" / "rope" / "thinking" / "final" / "checkpoint.pt",
    DRIVE_RUN_DIR / "analysis" / "aligned_ncc" / "selected_confirmation_summary.csv",
    DRIVE_RUN_DIR / "tables" / "training_sampling_distribution.csv",
    DRIVE_RUN_DIR / "tables" / "training_set_count_sampler_plan.csv",
    DRIVE_RUN_DIR / "tables" / "trace_readout_summary.csv",
    DRIVE_RUN_DIR / "tables" / "trace_readout_by_count.csv",
    DRIVE_RUN_DIR / "analysis" / "trace_readout" / "manifest.json",
]
missing = [str(path) for path in required_drive_files if not path.exists()]
assert not missing, "Drive persistence check failed: " + str(missing)
print("Drive persistence verified:", DRIVE_RUN_DIR)

if AUTO_DISCONNECT and Path("/content").exists():
    print(f"Disconnecting in {DISCONNECT_DELAY_SECONDS}s after successful persistence check...")
    time.sleep(DISCONNECT_DELAY_SECONDS)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        os.kill(os.getpid(), 9)